In [1]:
!pip install yfinance mftool

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for mftool: filename=mftool-3.2-py3-none-any.whl size=116921 sha256=09b82150716a3c625c5e14a674e68796003fdec90ac4c9fedd914b0f1dea902e
  Stored in directory: /root/.cache/pip/wheels/90/05/35/68405040c94b9f6eb5b6617714833f938a57a41784c2aabd4c
Successfully built mftool


In [2]:
import yfinance as yf
import pandas as pd
from mftool import Mftool

def get_stock_data(ticker: str, period: str = "1y") -> pd.DataFrame:
    df = yf.download(ticker, period=period, auto_adjust=True)
    df.dropna(inplace=True)
    return df

def get_stock_info(ticker: str) -> dict:
    stock = yf.Ticker(ticker)
    return stock.info

def get_mutual_fund_data(scheme_code: str):
    mf = Mftool()
    return mf.get_scheme_historical_nav(scheme_code, as_Dataframe=True)

def get_all_mf_schemes() -> dict:
    mf = Mftool()
    return mf.get_scheme_codes()

def enrich_stock_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["daily_return"] = df["Close"].pct_change()
    df["volatility_30d"] = df["daily_return"].rolling(30).std() * (252 ** 0.5)
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()
    return df.dropna()

In [3]:
df = get_stock_data("RELIANCE.NS", "3mo")
print("Stock data loaded:", df.shape)
print(df.tail(3))

[*********************100%***********************]  1 of 1 completed

Stock data loaded: (61, 5)
Price             Close         High          Low         Open      Volume
Ticker      RELIANCE.NS  RELIANCE.NS  RELIANCE.NS  RELIANCE.NS RELIANCE.NS
Date                                                                      
2026-03-30  1343.900024  1365.000000  1334.099976  1335.000000    24393861
2026-04-01  1369.199951  1384.400024  1362.900024  1384.199951    14404619
2026-04-02  1350.500000  1358.199951  1328.000000  1357.000000    21280051


In [5]:
df_enriched = enrich_stock_data(df)
print("Enriched data:", df_enriched.columns.tolist())
print(df_enriched[["Close", "daily_return", "volatility_30d", "SMA_20", "SMA_50"]].tail(3))

Enriched data: [('Close', 'RELIANCE.NS'), ('High', 'RELIANCE.NS'), ('Low', 'RELIANCE.NS'), ('Open', 'RELIANCE.NS'), ('Volume', 'RELIANCE.NS'), ('daily_return', ''), ('volatility_30d', ''), ('SMA_20', ''), ('SMA_50', '')]
Price             Close daily_return volatility_30d       SMA_20       SMA_50
Ticker      RELIANCE.NS                                                      
Date                                                                         
2026-03-30  1343.900024    -0.003115       0.252945  1390.585004  1410.420005
2026-04-01  1369.199951     0.018826       0.254326  1389.350000  1408.628003
2026-04-02  1350.500000    -0.013658       0.253572  1388.975000  1406.480002


In [6]:
info = get_stock_info("TCS.NS")
print("Stock info keys:", list(info.keys())[:10])
print("Sector:", info.get("sector"))
print("Market Cap:", info.get("marketCap"))

Stock info keys: ['address1', 'address2', 'city', 'zip', 'country', 'phone', 'fax', 'website', 'industry', 'industryKey']
Sector: Technology
Market Cap: 8866846736384


In [7]:
nav = get_mutual_fund_data("119551")
print("Mutual fund NAV loaded")
print(nav.head(3))

Mutual fund NAV loaded
                  nav  dayChange
date                            
02-04-2026  103.88130    -0.1303
31-03-2026  104.01160     0.0195
30-03-2026  103.99210     0.0819
